# EA3 - Actividad 3.1c: Streaming Sin Kafka (Modo Offline)

## Objetivos
- Simular un flujo de datos en tiempo real sin necesidad de Kafka
- Leer archivos JSONL linea por linea con delays controlados
- Aplicar procesamiento incremental a los datos que llegan
- Entender la diferencia entre procesar un archivo completo vs un flujo
- Visualizar metricas que evolucionan a medida que llegan datos nuevos

> **NOTA:** Este notebook funciona con el perfil **basico** de Docker.
> NO necesitas Kafka, Zookeeper ni Hive para ejecutarlo.
>
> ```bash
> docker-compose --profile basico up -d
> ```
>
> Esto lo hace ideal para quienes no lograron levantar el perfil completo.

## ¿Por que simular streaming?

En un entorno educativo, no siempre todos los estudiantes logran levantar el perfil completo de Docker (Kafka + Zookeeper + Hive).

La **simulacion de streaming** desde archivos JSONL te permite:

1. **Practicar los mismos conceptos** de procesamiento incremental sin infraestructura compleja
2. **Controlar la velocidad** de llegada de datos para entender el comportamiento
3. **Comparar batch vs streaming** usando exactamente los mismos datos
4. **Depurar** tu codigo de procesamiento antes de usarlo en Kafka real

### Flujo de trabajo

```
  Archivo JSONL          Lector Lineal           Procesador             Dashboard
  (1000 eventos)         (1 linea cada N ms)     (acumula metricas)     (se actualiza)
  +----------+           +----------------+       +---------------+      +-----------+
  | evento 1 | ---->    | readline()     | ----> | count += 1   | ----> | mostrar() |
  | evento 2 | ---->    | time.sleep()   | ----> | sum += monto | ----> | graficos  |
  | evento 3 | ---->    | readline()     | ----> | avg  = sum/n | ----> | metricas  |
  | ...      |           +----------------+       +---------------+      +-----------+
  +----------+
```

## Setup

In [ ]:
import json
import time
import os
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from IPython.display import clear_output, display

print("Librerias cargadas correctamente.")

# Verificar que existen datos de streaming
ruta_datos = "/home/jovyan/datos/streaming"
archivos = [f for f in os.listdir(ruta_datos) if f.endswith('.jsonl')]
print(f"Archivos JSONL disponibles en {ruta_datos}:")
for a in sorted(archivos):
    tam = os.path.getsize(os.path.join(ruta_datos, a))
    print(f"  {a:40s} {tam:>8,} bytes")

## 1. Funcion para Simular Streaming desde JSONL

La funcion `simular_streaming` lee un archivo JSONL **linea por linea**, con un delay configurable entre eventos.

Esto replica exactamente como llegarian los datos desde Kafka: **uno a la vez, en orden**.

In [ ]:
def simular_streaming(archivo, procesar_evento, delay_ms=200, max_eventos=None):
    """
    Simula un stream de datos leyendo un archivo JSONL linea por linea.

    Parametros:
        archivo (str): Ruta al archivo JSONL
        procesar_evento (function): Funcion que recibe cada evento como dict
        delay_ms (int): Milisegundos entre cada evento (default: 200)
        max_eventos (int): Maximo de eventos a procesar (default: None = todos)

    Retorna:
        total_eventos (int): Cuantos eventos se procesaron
        tiempo_total (float): Tiempo total en segundos
    """
    delay_seg = delay_ms / 1000.0
    total_eventos = 0
    inicio = time.time()

    print(f"Iniciando stream desde: {archivo}")
    print(f"Delay entre eventos: {delay_ms} ms")
    print(f"Max eventos: {max_eventos or 'todos'}")
    print(f"{'='*50}")

    with open(archivo, "r") as f:
        for linea in f:
            if max_eventos and total_eventos >= max_eventos:
                break

            linea = linea.strip()
            if not linea:
                continue

            evento = json.loads(linea)
            procesar_evento(evento, total_eventos + 1)
            total_eventos += 1
            time.sleep(delay_seg)

    tiempo_total = time.time() - inicio
    return total_eventos, tiempo_total

## 2. Ejemplo Simple: Contar Eventos en Tiempo Real

Vamos a simular la llegada de 20 transacciones con un delay de 500ms cada una, solo contando cuantas llegan.

In [ ]:
contador = 0

def contar_evento(evento, seq):
    global contador
    contador += 1
    print(f"[{seq:3d}] Evento recibido: {evento['tipo_evento']:20s} | "
          f"ID: {evento['id']:15s} | "
          f"Total acumulado: {contador}")

total, tiempo = simular_streaming(
    "/home/jovyan/datos/streaming/transacciones_1000.jsonl",
    contar_evento, delay_ms=500, max_eventos=20
)

print(f"\n{'='*50}")
print(f"Stream finalizado: {total} eventos en {tiempo:.1f} segundos")

## 3. Procesamiento Incremental con Acumuladores

La clave del streaming es **actualizar metricas incrementalmente** a medida que llegan los datos, sin tener que reprocesar todo.

Veamos como mantener un contador, suma y promedio en tiempo real:

In [ ]:
class ProcesadorStreaming:
    """Acumula metricas de transacciones en tiempo real."""

    def __init__(self):
        self.total_eventos = 0
        self.monto_acumulado = 0.0
        self.productos = defaultdict(lambda: {"cantidad": 0, "monto": 0.0})
        self.metodos_pago = defaultdict(int)
        self.montos_historicos = []

    def procesar(self, evento, seq):
        self.total_eventos += 1
        monto = evento.get("monto", 0) or 0
        self.monto_acumulado += monto
        self.montos_historicos.append(monto)

        producto = evento.get("producto", "desconocido") or "desconocido"
        self.productos[producto]["cantidad"] += 1
        self.productos[producto]["monto"] += monto

        metodo = evento.get("metodo_pago", "desconocido") or "desconocido"
        self.metodos_pago[metodo] += 1

    def metricas(self):
        return {
            "eventos": self.total_eventos,
            "monto_total": self.monto_acumulado,
            "promedio": self.monto_acumulado / self.total_eventos if self.total_eventos > 0 else 0,
            "productos": dict(self.productos),
            "metodos_pago": dict(self.metodos_pago),
        }

    def mostrar_resumen(self):
        m = self.metricas()
        print(f"  Eventos:    {m['eventos']:5d}")
        print(f"  Monto tot:  ${m['monto_total']:>10,.0f}")
        print(f"  Promedio:   ${m['promedio']:>10,.0f}")

print("Clase ProcesadorStreaming definida.")

In [ ]:
procesador = ProcesadorStreaming()

def procesar_y_mostrar(evento, seq):
    procesador.procesar(evento, seq)
    clear_output(wait=True)
    print(f"Procesando evento #{seq}...")
    print(f"  {evento['producto']:20s} | ${evento.get('monto',0):>8,} | {evento.get('metodo_pago','')}")
    print()
    procesador.mostrar_resumen()

total, tiempo = simular_streaming(
    "/home/jovyan/datos/streaming/transacciones_1000.jsonl",
    procesar_y_mostrar, delay_ms=200, max_eventos=30
)

print(f"\n{'='*50}")
print(f"Stream finalizado: {total} eventos en {tiempo:.1f}s")

## 4. Dashboard Evolutivo en Tiempo Real

Ahora combinamos todo: simulamos un flujo de datos y actualizamos graficos en vivo que muestran como evolucionan las metricas.

In [ ]:
class DashboardStreaming:
    """Dashboard que se actualiza en vivo al recibir eventos."""

    def __init__(self):
        self.procesador = ProcesadorStreaming()
        self.historial_monto = []
        self.historial_eventos = []

    def procesar(self, evento, seq):
        self.procesador.procesar(evento, seq)
        self.historial_eventos.append(seq)
        self.historial_monto.append(self.procesador.monto_acumulado)

    def mostrar(self, iteracion):
        clear_output(wait=True)
        m = self.procesador.metricas()

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f"Dashboard Streaming en Vivo - Evento #{iteracion}",
                     fontsize=14, fontweight='bold')

        # Grafico 1: Ventas por producto (barras)
        if m['productos']:
            df_prod = pd.DataFrame([
                (p, d['monto']) for p, d in m['productos'].items()
            ], columns=['producto', 'monto']).sort_values('monto', ascending=True)
            axes[0, 0].barh(df_prod['producto'], df_prod['monto'], color='steelblue')
        axes[0, 0].set_title('Ventas por Producto')
        axes[0, 0].set_xlabel('Total ($)')

        # Grafico 2: Metodos de pago
        if m['metodos_pago']:
            df_pago = pd.DataFrame(
                list(m['metodos_pago'].items()),
                columns=['metodo', 'count']
            )
            axes[0, 1].bar(df_pago['metodo'], df_pago['count'],
                          color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'])
        axes[0, 1].set_title('Transacciones por Metodo de Pago')
        axes[0, 1].tick_params(axis='x', rotation=45)

        # Grafico 3: Monto acumulado en el tiempo
        if len(self.historial_monto) > 1:
            axes[1, 0].plot(self.historial_eventos, self.historial_monto,
                          color='green', linewidth=2)
            axes[1, 0].fill_between(self.historial_eventos, self.historial_monto,
                                   alpha=0.1, color='green')
        axes[1, 0].set_title('Monto Acumulado')
        axes[1, 0].set_xlabel('Evento #')
        axes[1, 0].set_ylabel('Total ($)')

        # Grafico 4: Metricas clave
        axes[1, 1].axis('off')
        texto = (
            f"Eventos procesados:\n{m['eventos']}\n\n"
            f"Monto total:\n${m['monto_total']:,.0f}\n\n"
            f"Ticket promedio:\n${m['promedio']:,.0f}\n\n"
            f"Productos distintos:\n{len(m['productos'])}"
        )
        axes[1, 1].text(0.5, 0.5, texto,
                       transform=axes[1, 1].transAxes,
                       fontsize=14, verticalalignment='center',
                       horizontalalignment='center',
                       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        axes[1, 1].set_title('Metricas Clave')

        plt.tight_layout()
        plt.show()

print("DashboardStreaming listo.")

In [ ]:
dashboard = DashboardStreaming()

def procesar_y_graficar(evento, seq):
    dashboard.procesar(evento, seq)
    if seq % 5 == 0:
        dashboard.mostrar(seq)

total, tiempo = simular_streaming(
    "/home/jovyan/datos/streaming/transacciones_1000.jsonl",
    procesar_y_graficar,
    delay_ms=300,
    max_eventos=50
)

print(f"\nStream finalizado: {total} eventos en {tiempo:.1f}s")

## 5. Diferencia Clave: Batch vs Streaming

Una pregunta fundamental: **?Que cambia si procesamos todo el archivo de una vez (batch) vs linea por linea (streaming)?**

Vamos a demostrarlo con exactamente los mismos datos:

In [ ]:
archivo = "/home/jovyan/datos/streaming/transacciones_1000.jsonl"

# --- MODO BATCH: Leer todo de una vez ---
inicio_batch = time.time()
with open(archivo) as f:
    datos_batch = [json.loads(linea) for linea in f]
tiempo_batch = time.time() - inicio_batch

monto_total_batch = sum(e.get("monto", 0) or 0 for e in datos_batch)

print("=== MODO BATCH (todo de una vez) ===")
print(f"Eventos:     {len(datos_batch)}")
print(f"Monto total: ${monto_total_batch:,.0f}")
print(f"Tiempo:      {tiempo_batch:.4f}s")
print()

# --- MODO STREAMING: Leer linea por linea ---
procesador_stream = ProcesadorStreaming()

inicio_stream = time.time()
with open(archivo) as f:
    for i, linea in enumerate(f, 1):
        evento = json.loads(linea.strip())
        procesador_stream.procesar(evento, i)
tiempo_stream = time.time() - inicio_stream

m = procesador_stream.metricas()

print("=== MODO STREAMING (linea por linea) ===")
print(f"Eventos:     {m['eventos']}")
print(f"Monto total: ${m['monto_total']:,.0f}")
print(f"Tiempo:      {tiempo_stream:.4f}s")
print()

# El resultado de las metricas debe ser identico
print(f"{" Resultado identico ":=^50}")

### Conclusiones:

| Aspecto | Batch | Streaming |
|---------|-------|-----------|
| **Datos disponibles** | Todos desde el inicio | Llegan uno a uno |
| **Tiempo de procesamiento** | Rapido (todo junto) | Lento (va pausado) |
| **Memoria** | Tiene todo en memoria | Solo un evento a la vez |
| **Latencia del resultado** | Tarda hasta tener todo | Resultados inmediatos |
| **Datos infinitos?** | No (debe tener fin) | Si (flujo continuo) |
| **Actualizacion de metricas** | Al final | En cada evento |

En el modo batch, procesamos los datos cuando ya estan todos disponibles.
En streaming, procesamos a medida que llegan, obteniendo **resultados parciales inmediatos**.

---
## Ejercicios

In [ ]:
# =============================================================
# EJERCICIO 1: Procesar logs en streaming
# =============================================================
# TODO: Usando logs_1000.jsonl, implementa un procesador que:
#
# 1. Lea el archivo en modo streaming (linea por linea)
#
# 2. Acumule:
#    - Total de eventos por status_code (200, 404, 500, etc.)
#    - Tiempo de respuesta promedio
#    - Total de errores (status >= 400)
#
# 3. Por cada 10 eventos, muestre un resumen con:
#    clear_output(wait=True)
#    "Logs procesados: X"
#    "Status 200: X | Errores 4xx: X | Errores 5xx: X"
#    "Response time promedio: X ms"
#
# 4. Al final, muestre un resumen completo
#
# Pistas:
#   - json.loads(linea) para cada linea
#   - Lleva un dict: conteo_status[status] += 1
#   - Para promedio: suma_response / total_eventos
#   - Usa: from IPython.display import clear_output

ARCHIVO_LOGS = "/home/jovyan/datos/streaming/logs_1000.jsonl"

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 2: Detectar anomalias IoT en streaming
# =============================================================
# TODO: Usando iot_dirty_500.jsonl (calidad BAJA, contiene errores):
#
# 1. Procesa cada evento en streaming
#
# 2. Detecta anomalias:
#    - Temperatura > 50°C (sensor danado)
#    - Humedad negativa (error de lectura)
#    - Sensor_id nulo (dato corrupto)
#
# 3. Lleva dos contadores:
#    - total_eventos
#    - total_anomalias (y cuales son)
#
# 4. Al final, muestra:
#    "De X eventos, se detectaron Y anomalias"
#    "Tasa de anomalia: X.X%"
#    "Desglose: tempe: X, humedad: Y, nulos: Z"
#
# Pistas:
#   - Verifica si el valor existe: evento.get("temperatura") is not None
#   - Temperatura alta: evento["temperatura"] > 50
#   - Humedad negativa: isinstance(valor, (int,float)) and valor < 0
#   - Sensor nulo: evento.get("sensor_id") is None

ARCHIVO_IOT = "/home/jovyan/datos/streaming/iot_dirty_500.jsonl"

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 3: Ventana deslizante (Sliding Window)
# =============================================================
# TODO: Implementa una ventana deslizante que mantenga
#   las metricas SOLO de los ultimos N eventos.
#
# Por ejemplo, con ventana de 10 eventos:
#   - Cuando llega el evento 11, el evento 1 "sale" de la ventana
#   - El promedio se recalcula con eventos 2..11
#
# Esto replica el concepto de window en Spark Structured Streaming.
#
# Instrucciones:
# 1. Lee transacciones_1000.jsonl en streaming
# 2. Manten una lista de los ultimos 10 montos
# 3. Por cada nuevo evento, agrega el monto y si hay mas de 10,
#    elimina el mas antiguo
# 4. Muestra: "Evento X | Ventana: [monto1, monto2,...] | Prom: $X"
#
# Pistas:
#   - ventana = []
#   - ventana.append(monto)
#   - if len(ventana) > 10: ventana.pop(0)
#   - promedio = sum(ventana) / len(ventana)

ARCHIVO_TX = "/home/jovyan/datos/streaming/transacciones_1000.jsonl"
TAMANO_VENTANA = 10

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **Simular streaming** desde archivos JSONL sin necesidad de Kafka
2. **Procesamiento incremental:** Acumular metricas evento por evento
3. **Dashboard evolutivo:** Visualizar como cambian las metricas a medida que llegan datos
4. **Batch vs Streaming:** Mismos datos, enfoques diferentes, resultados identicos al final
5. **Ventanas deslizantes:** Mantener metricas solo de los ultimos N eventos
6. **Deteccion de anomalias:** Identificar datos corruptos o fuera de rango en un flujo

Este conocimiento se aplica directamente cuando trabajas con Kafka + Spark Structured Streaming.

---
## Desafio Extra (Opcional)

**Simulador de Kafka con productor en background:**

Combina lo aprendido: crea un "mock de Kafka" que:

1. Use `threading` para leer el JSONL en un hilo separado (el "productor")
2. Use una cola (`queue.Queue`) para pasar eventos entre hilos
3. En el hilo principal, consuma eventos de la cola (el "consumidor")
4. El consumidor puede procesar mas lento o mas rapido que el productor
5. La cola actua como el buffer de Kafka

In [ ]:
# =============================================================
# DESAFIO: Mock de Kafka con threading + Queue
# =============================================================
# TODO: Implementa un sistema productor-consumidor usando:
#   - threading.Thread para el productor
#   - queue.Queue como buffer
#   - El hilo principal como consumidor
#
# Pistas:
#   import threading, queue
#   cola = queue.Queue(maxsize=50)
#   productor = threading.Thread(target=...)
#   productor.start()
#   while cola.qsize() > 0 or productor.is_alive():
#       evento = cola.get(timeout=1)
#       # procesar evento
#
# Escribe tu codigo aqui:
